In [11]:
import pandas as pd
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [12]:
# load data
from pathlib import Path

# Resolve repo root regardless of execution cwd.
def find_repo_root(start: Path) -> Path:
    for path in [start] + list(start.parents):
        if (path / "data" / "label.csv").exists():
            return path
    raise FileNotFoundError("Could not find data/label.csv")

repo_root = find_repo_root(Path.cwd())
sitedata_path = repo_root / "data" / "label.csv"
sitedata = pd.read_csv(sitedata_path, sep=",")

In [13]:
aglen = sitedata.iloc[:,1]
ablen = sitedata.iloc[:,2]

label = []
for i in range(0,len(aglen)):
    interaction_matrix = np.zeros((aglen[i],ablen[i]))
    # load site data
    interaction_sites = sitedata.iloc[:,3][i]
    interaction_sites = eval(interaction_sites) 
    

    # set interaction site = 1
    for site in interaction_sites:
        interaction_matrix[site[0]-1, site[1]-1] = 1
     
    label.append(interaction_matrix)  

In [14]:
from torch.nn.utils.rnn import pad_sequence

label_y_tensor = [torch.tensor(matrix, device=device) for matrix in label]

max_rows = max(matrix.shape[0] for matrix in label_y_tensor)
max_cols = max(matrix.shape[1] for matrix in label_y_tensor)
print('max_rows: ', max_rows)
print('max_cols: ', max_cols)
padded_label_y_tensor = [torch.nn.functional.pad(matrix, (0, max_cols - matrix.shape[1])) for matrix in label_y_tensor]
padded_label_y_tensor = pad_sequence(padded_label_y_tensor, batch_first=True, padding_value=0)

print(padded_label_y_tensor.shape)

# save label tensor file
output_dir = repo_root / "data"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "label_y.pt"
torch.save(padded_label_y_tensor.cpu(), output_path)
print('successfully save the representation data!')

max_rows:  498
max_cols:  263
torch.Size([4056, 498, 263])
successfully save the representation data!
